# FAST-001 — one automatic development launch

Enable Internet and two T4 GPUs, then **Run All**. No mode switching, Pilot file or duration-approval field is needed. The same run performs CPU functional preflight, independent data checks, a short same-GPU performance check, then the unchanged three-arm development experiment. Any failed check stops before training. Existing matching checkpoints are resumed; completed arms are reused. The runtime estimate is informational; no hard deadline or automatic budget reduction is imposed.

This user-authorized development workflow does not open confirmation data or claim scientific success.


In [ ]:
import json
import time
from pathlib import Path
import subprocess
import sys
import os
import importlib.util

SESSION_STARTED = time.perf_counter()
SOURCE_COMMIT = "1d393e7622821433712841c7ba64d7de689e7dc4"
REL = "research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/exploration-fast-001"
ROOT = Path("/kaggle/working/fast001-code-" + SOURCE_COMMIT[:12])
WORK = Path("/kaggle/working/fast001-output-" + SOURCE_COMMIT[:12])
if SOURCE_COMMIT.startswith("__"):
    raise RuntimeError("Import the committed Notebook with its pinned source")
ROOT.mkdir(parents=True, exist_ok=True)


def git(*args):
    print("[Setup] " + " ".join(args), flush=True)
    subprocess.run(["git", *args], cwd=ROOT, check=True)


if not (ROOT / ".git").exists():
    git("init", "-b", "fast001")
    git("remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git")
    git("sparse-checkout", "init", "--no-cone")
    git(
        "sparse-checkout",
        "set",
        "--no-cone",
        "/pyproject.toml",
        "/src/",
        "/shared/python/",
        "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/config/registered.json",
        "/research/governance/formal_seed_registry.json",
    )
    git("fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT)
    git("checkout", "--detach", "FETCH_HEAD")
if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError("Existing checkout source differs; preserve previous Output")
ENTRY = ROOT / REL / "run_fast001.py"
WORK.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "PYTHONUNBUFFERED": "1"}
def run(*args):
    subprocess.run([sys.executable, str(ENTRY), *args], cwd=ROOT, env=ENV, check=True)

In [ ]:
# Reuse the existing public-data credential if available; never print token values.
missing=[p for p in ('datasets','tokenizers','huggingface_hub') if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing],check=True)
if not ENV.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        ENV['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN').strip()
    except Exception:
        print('HF_TOKEN unavailable; using anonymous public dataset access',flush=True)
run('--mode','auto','--devices','cuda:0','cuda:1','--output',str(WORK))
print('Retain this Kaggle Output with Save Version:',WORK)
(WORK/'notebook-timing.json').write_text(json.dumps({'source_sha':SOURCE_COMMIT,
    'end_to_end_seconds':time.perf_counter()-SESSION_STARTED,'scientific_decision':False},indent=2)+'\n')


## Checkpoints and scope

All three arms retain 500 steps / 256,000 target tokens, FP32 and the original joint objective. S16 and S8 start concurrently; the first free GPU takes M0. Profiling and interventions start only after training workers exit. Save Version retains portable checkpoints for a later cloud session; `/working` alone is not persistent. There is no wall-time kill, timed pause or extra scientific arm. Diagnostic CLI modes remain available for engineering use, but are not required for a normal launch.
